# 01 - Plain WGAN-GP on NoC adjacency matrices

**Where this sits:** **`WGAN-GP`** -> RWGAN. This is the baseline generator: a
Wasserstein GAN with gradient penalty and **no reward signal**. It is pulled only
toward "look like a real 9-router topology", so it should learn the shape of the
whole training distribution (connected 9x9 symmetric adjacency matrices, router
degree <= 4).

Notebook `02` keeps this exact WGAN-GP core and adds a frozen connection-count
reward CNN that pulls the generator toward denser topologies.

## What this notebook does
1. Build a small in-memory dataset of valid topologies (paper "Algorithm 1").
2. Train a WGAN-GP with `gannoc.training.train_wgangp`, which pins `LAMBDA` at
   1.0 so the reward branch is never used.
3. Evaluate the trained generator: structural validity, novelty, connection count.
4. Draw a few generated topologies with `networkx`.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../src")  # use the repo's own gannoc package

import pickle
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import tensorflow as tf

from gannoc import data as gdata
from gannoc import training
from gannoc.evaluate import evaluate_generator

OUT = "output"  # demos/output/ (git-ignored)

## 1. A tiny training set

`gannoc.data.generate_dataset` builds valid topologies by random construction
(RAPIDO 2021 / PhD thesis Chapter 5, "Algorithm 1") -- connected 9x9 symmetric
binary matrices, zero diagonal, every router degree <= 4, stratified by physical
connection count. Pure Python, no simulator. We keep it small for a fast run.

In [ ]:
raw = gdata.generate_dataset(
    connection_counts=range(8, 19),  # a spanning tree (8) up to degree-4-saturated (18)
    samples_per_class=200,
    seed=0,
)
gdata.save_dataset(raw, f"{OUT}/_demo_nocs.npz")

dataset = gdata.load_dataset(f"{OUT}/_demo_nocs.npz")
print(f"{len(dataset)} topologies, {dataset.n_routers} routers")
print("images  :", dataset.images().shape, dataset.images().dtype)
print("conn/sec:", np.unique(dataset.n_connections))

## 2. Train the WGAN-GP

`gannoc.training.train_wgangp` runs the WGAN-GP loop with `LAMBDA` pinned at
1.0, so the generator loss is **only** the Wasserstein term and no reward
network is needed.

Every hyper-parameter is a module constant in `gannoc/training.py` (the paper's
values -- see its "Parameter provenance" docstring). This demo assigns a couple
of them directly to keep the run short; `epochs` is the one knob the entry point
itself takes.

In [ ]:
# Demo overrides of gannoc.training's module constants (paper values otherwise).
training.SEED = 0
training.EVAL_EVERY = 5

result = training.train_wgangp(
    dataset_path=f"{OUT}/_demo_nocs.npz",
    output_dir=OUT,
    run_name="01_wgan_gp",
    epochs=5,          # smoke run; raise for a cleaner generator
)
with open(result["history"], "rb") as fh:
    history = pickle.load(fh)
print("generator saved to:", result["generator_checkpoint"])
print("history keys      :", list(history.keys()))

## 3. Evaluate the generator

`evaluate_generator` samples the generator, binarizes + symmetrizes each output
(`gannoc.data.binarize_symmetric`), and measures: the fraction that are
structurally **valid** (connected, degree <= 4, all routers wired), the fraction
**novel** vs the training set, and the connection-count distribution.

In [ ]:
generator = tf.keras.models.load_model(result["generator_checkpoint"], compile=False)
metrics = evaluate_generator(generator, dataset, n_samples=1000, seed=0)
print(metrics.summary())

## 4. Look at a few generated topologies

After only 5 epochs the generator is far from converged: expect a mix of
connected degree-<=4 graphs and still-invalid ones.

In [ ]:
latent_dim = generator.input_shape[-1]
z = np.random.default_rng(0).random((6, latent_dim)).astype("float32")  # uniform, as in training
rawgen = generator.predict(z, verbose=0)[..., 0]  # (6, 9, 9) in [-1, 1]
mats = [gdata.binarize_symmetric(m, threshold=0.0) for m in rawgen]

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, m in zip(axes.ravel(), mats):
    g = nx.from_numpy_array(np.asarray(m))
    ok = gdata.is_valid_topology(m)
    nx.draw_circular(g, ax=ax, node_size=250, node_color="#cfe8ff", with_labels=True)
    ax.set_title(f"{gdata.n_connections_of(m)} links - " + ("valid" if ok else "invalid"))
    ax.set_axis_off()
fig.tight_layout()
fig.savefig(f"{OUT}/01_topology_samples.png", dpi=120)
plt.show()

## Result

The WGAN-GP learns the *shape* of the topology distribution but is given no
reason to prefer any particular connection count -- its generated topologies
spread across the 8..18 range roughly like the training set. Notebook `02` adds
a frozen reward CNN and anneals `LAMBDA` so that spread is pushed denser.